# Forecasting next-day and next-week AQI categories with Random Forest

**Unit of prediction.** One row per station-day (about 17,000 rows), not hourly rows carrying a repeated daily label.

**Inputs — all known by the end of day *t*.** Daily mean pollutants and weather for day *t*, for yesterday, and their 3- and 7-day rolling means; day *t*'s category labels; the station; and the calendar position of the **target** day (month and day of year, known in advance). Nothing from the target day itself is used.

**Targets.** Each category in `TARGET_COLUMNS` at *t+1* (**next day**) and *t+7* (**next week**, i.e. the category 7 days ahead).

**Split.** Chronological: the final 20% of dates form the test period, and training rows whose *target* date falls in that period are purged, so no training label comes from the test period. No random shuffling anywhere.

**Baseline.** *Persistence* — "the future looks like today". Every model is scored against it on exactly the same test rows, because a forecast is only useful if it beats persistence.

**Data.** The daily `aqi_daily` MySQL table, with China AQI breakpoints, validity gates and category rules coming from the shared `aqi_labels.py` — the same module `sets.ipynb` uses, so the two notebooks cannot drift apart.

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display
from sqlalchemy import create_engine, text

try:
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import (
        accuracy_score, balanced_accuracy_score, classification_report,
        confusion_matrix, f1_score,
    )
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder
except ImportError as error:
    raise ImportError(
        "This modeling notebook requires scikit-learn. Install it in this notebook's "
        "Python environment with `%pip install scikit-learn`, then restart the kernel."
    ) from error

# aqi_labels.py holds the China AQI rules, shared with sets.ipynb.
NOTEBOOK_DIR = Path.cwd()
helper_files = list(NOTEBOOK_DIR.rglob("aqi_labels.py"))
if not helper_files:
    raise FileNotFoundError(
        f"aqi_labels.py was not found under {NOTEBOOK_DIR}. "
        "Copy the shared aqi_labels.py file into this folder or one of its subfolders."
    )

helper_dir = helper_files[0].parent.resolve()
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from aqi_labels import (  # noqa: E402
    AQI_CATEGORIES,
    O3_1H_BREAKPOINTS,
    POLLUTANTS,
    POLLUTANT_BREAKPOINTS,
    TARGET_COLUMNS,
    category_codes,
    interpolate_iaqi,
)

RANDOM_STATE = 42
TEST_FRACTION = 0.20                      # final 20% of dates are the test period
HORIZONS = {"next_day": 1, "next_week": 7}  # days ahead of day t
SHOW_DETAILS = True                       # confusion matrices / per-class reports
SOURCE_TABLE = "aqi_daily"
EXPECTED_SOURCE_ROWS = 51_920
DB_URL = os.environ.get("AIR_QUALITY_DB_URL", "mysql+pymysql://root:@localhost/air_quality")

## 1. Load the daily `aqi_daily` table

In [ ]:
if not re.fullmatch(r"[A-Za-z0-9_]+", SOURCE_TABLE):
    raise ValueError(f"Invalid MySQL table name: {SOURCE_TABLE!r}")
engine = create_engine(DB_URL, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        source_df = pd.read_sql_query(text(f"SELECT * FROM {SOURCE_TABLE}"), connection)
except Exception as error:
    raise RuntimeError(
        f"Could not load {SOURCE_TABLE!r} from the air_quality database. Check MySQL, "
        "AIR_QUALITY_DB_URL, the table name, SQLAlchemy, and PyMySQL. "
        f"Original database error: {error}"
    ) from error

print(f"Loaded source rows: {len(source_df):,}")
if len(source_df) != EXPECTED_SOURCE_ROWS:
    print(f"Note: expected {EXPECTED_SOURCE_ROWS:,} rows, found {len(source_df):,}; continuing after schema checks.")
required=["date","station","station_id",*POLLUTANTS,"O3_1h_max","TEMP_mean","TEMP_range","PRES_mean","DEWP_mean","RH_mean","RAIN_sum","WSPM_mean","wd_sin","wd_cos"]
missing=[column for column in required if column not in source_df]
if missing: raise KeyError(f"air_quality.{SOURCE_TABLE} is missing columns: {missing}")
if source_df.duplicated(["station_id","date"]).any(): raise ValueError("aqi_daily contains duplicate station-days")
display(source_df.head())

## 2. Daily China AQI and pollutant category labels

Same shared `build_daily_labels` as `sets.ipynb`, so the breakpoints, validity gates and category boundaries are unchanged. The output is a station-day table. These labels are used as **future targets** and as "today's category" features, never as same-day targets for same-day features.

In [ ]:
# Recompute daily AQI and individual pollutant categories from the database's daily concentrations.
def build_labels_from_daily(frame):
    daily=frame.copy()
    daily["date"]=pd.to_datetime(daily["date"],errors="coerce").dt.normalize()
    iaqi_columns=[]
    for pollutant in POLLUTANTS:
        values=pd.to_numeric(daily[pollutant],errors="coerce")
        col=f"{pollutant}_iaqi"
        if pollutant=="O3":
            sub=interpolate_iaqi(values,POLLUTANT_BREAKPOINTS["O3"])
            fallback=interpolate_iaqi(pd.to_numeric(daily["O3_1h_max"],errors="coerce"),O3_1H_BREAKPOINTS)
            mask=values.gt(800)
            sub.loc[mask]=fallback.loc[mask]
        else:
            sub=interpolate_iaqi(values,POLLUTANT_BREAKPOINTS[pollutant])
        daily[col]=sub.round()
        daily[f"{pollutant}_category"]=category_codes(daily[col]).map(AQI_CATEGORIES)
        iaqi_columns.append(col)
    daily["aqi"]=daily[iaqi_columns].max(axis=1,skipna=True)
    daily["aqi_category_code"]=category_codes(daily["aqi"])
    daily["aqi_category"]=daily["aqi_category_code"].map(AQI_CATEGORIES)
    daily["main_pollutant"]=daily[iaqi_columns].fillna(-1).idxmax(axis=1).str.removesuffix("_iaqi")
    daily.loc[daily[iaqi_columns].isna().all(axis=1),"main_pollutant"]=np.nan
    return daily

daily_labels=build_labels_from_daily(source_df)
print(f"Daily station records: {len(daily_labels):,}")
display(daily_labels[TARGET_COLUMNS].notna().sum().rename("valid_station_days").to_frame())


## 3. Build one row per station-day with history features

Hourly readings are aggregated to daily means (rain is summed). Each station is then placed on a **complete daily calendar**, so a missing day stays a gap instead of silently shifting "yesterday" or "next week" to the wrong date. Every feature uses only day *t* and earlier.

In [ ]:
# The database table is already daily, so use its pollutant and ERA5 summaries directly.
WEATHER_COLUMNS=["TEMP_mean","TEMP_range","PRES_mean","DEWP_mean","RH_mean","RAIN_sum","WSPM_mean","wd_sin","wd_cos"]
MEASURED_COLUMNS=[*POLLUTANTS,*WEATHER_COLUMNS]
daily=daily_labels[["station_id","station","date",*MEASURED_COLUMNS,*TARGET_COLUMNS]].copy()
daily["date"]=pd.to_datetime(daily["date"],errors="coerce").dt.normalize()
for column in MEASURED_COLUMNS:
    daily[column]=pd.to_numeric(daily[column],errors="coerce")
for target in TARGET_COLUMNS:
    daily[target]=daily[target].astype(object).where(daily[target].notna(),np.nan)

# Complete each station's calendar so lags and rolling windows reflect elapsed days.
def complete_calendar(frame):
    pieces=[]
    for station,group in frame.groupby("station"):
        group=group.sort_values("date").set_index("date")
        full=pd.date_range(group.index.min(),group.index.max(),freq="D")
        group=group.reindex(full)
        group["station"]=station
        group.index.name="date"
        pieces.append(group.reset_index())
    return pd.concat(pieces,ignore_index=True)

daily=complete_calendar(daily).sort_values(["station","date"]).reset_index(drop=True)
by_station=daily.groupby("station")
history={}
for column in MEASURED_COLUMNS:
    history[f"{column}_d0"]=daily[column]
    history[f"{column}_lag1"]=by_station[column].shift(1)
    history[f"{column}_roll3"]=by_station[column].transform(lambda s:s.rolling(3,min_periods=1).mean())
    history[f"{column}_roll7"]=by_station[column].transform(lambda s:s.rolling(7,min_periods=1).mean())
history=pd.DataFrame(history,index=daily.index)
today_labels=daily[TARGET_COLUMNS].add_prefix("today_")
feature_base=pd.concat([daily[["station"]],history,today_labels],axis=1)
HISTORY_FEATURES=list(history.columns)
TODAY_FEATURES=list(today_labels.columns)
print(f"Station-days in complete daily calendar: {len(daily):,}")
print(f"Date range: {daily.date.min().date()} to {daily.date.max().date()}")
print(f"History features: {len(HISTORY_FEATURES)} numeric + {len(TODAY_FEATURES)} current-day categories + station")


## 4. Chronological split with purging

Dates are ordered and the last `TEST_FRACTION` of them become the test period. For a horizon of *h* days:

- **Test rows:** forecast origin *t* on or after the cutoff.
- **Training rows:** target date *t+h* strictly **before** the cutoff, so even the training *labels* never come from the test period.

Random shuffling is not used anywhere.

In [ ]:
all_dates = np.sort(daily["date"].unique())
cutoff = pd.Timestamp(all_dates[int(len(all_dates) * (1 - TEST_FRACTION))])
print(f"Training origins/targets before {cutoff.date()}  |  test origins from {cutoff.date()} "
      f"to {pd.Timestamp(all_dates[-1]).date()}")


def calendar_features(dates):
    # Calendar position of the TARGET day: known in advance, so not leakage.
    doy = dates.dt.dayofyear
    month = dates.dt.month
    return pd.DataFrame({
        "target_month_sin": np.sin(2 * np.pi * (month - 1) / 12),
        "target_month_cos": np.cos(2 * np.pi * (month - 1) / 12),
        "target_doy_sin": np.sin(2 * np.pi * (doy - 1) / 365.25),
        "target_doy_cos": np.cos(2 * np.pi * (doy - 1) / 365.25),
    }, index=dates.index)


CALENDAR_FEATURES = list(calendar_features(daily["date"].head(5)).columns)
NUMERIC_FEATURES = HISTORY_FEATURES + CALENDAR_FEATURES
CATEGORICAL_FEATURES = ["station"] + TODAY_FEATURES
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

## 5. Train and evaluate: Random Forest vs. persistence baseline

One classifier per (horizon, target). Preprocessing is fit on training rows only. Each model is scored next to the persistence baseline on **exactly the same test rows** (rows where today's category and the future category both exist).

In [ ]:
forecast_models = {}
rows = []

for horizon_name, horizon_days in HORIZONS.items():
    target_date = daily["date"] + pd.Timedelta(days=horizon_days)
    X_all = pd.concat([feature_base, calendar_features(target_date)], axis=1)[FEATURE_COLUMNS]

    for target in TARGET_COLUMNS:
        y_future = daily.groupby("station")[target].shift(-horizon_days)
        today = daily[target]

        train_mask = y_future.notna() & (target_date < cutoff)
        test_mask = y_future.notna() & today.notna() & (daily["date"] >= cutoff)
        if train_mask.sum() == 0 or test_mask.sum() == 0:
            print(f"Skipping {horizon_name}/{target}: no labeled train or test rows.")
            continue

        y_train = y_future[train_mask].astype(str)
        y_test = y_future[test_mask].astype(str)
        if y_train.nunique() < 2:
            print(f"Skipping {horizon_name}/{target}: fewer than two classes in training.")
            continue
        X_train, X_test = X_all[train_mask], X_all[test_mask]
        baseline = today[test_mask].astype(str)  # persistence: "it will be like today"

        preprocessing = ColumnTransformer([
            ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), NUMERIC_FEATURES),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), CATEGORICAL_FEATURES),
        ])
        model = Pipeline([
            ("preprocessing", preprocessing),
            ("classifier", RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        forecast_models[(horizon_name, target)] = model

        row = {
            "horizon": horizon_name,
            "target": target,
            "train_rows": int(train_mask.sum()),
            "test_rows": int(test_mask.sum()),
            "rf_accuracy": accuracy_score(y_test, predictions),
            "persistence_accuracy": accuracy_score(y_test, baseline),
            "rf_balanced_acc": balanced_accuracy_score(y_test, predictions),
            "persistence_balanced_acc": balanced_accuracy_score(y_test, baseline),
            "rf_macro_f1": f1_score(y_test, predictions, average="macro", zero_division=0),
            "persistence_macro_f1": f1_score(y_test, baseline, average="macro", zero_division=0),
        }
        row["accuracy_gain_vs_persistence"] = row["rf_accuracy"] - row["persistence_accuracy"]
        rows.append(row)

        if SHOW_DETAILS:
            labels = sorted(set(y_train) | set(y_test) | set(predictions))
            print(f"\n=== {horizon_name} (t+{horizon_days}d) | {target} ===")
            print(f"Train rows: {row['train_rows']:,} | Test rows: {row['test_rows']:,}")
            print(f"RF accuracy {row['rf_accuracy']:.3f} vs persistence {row['persistence_accuracy']:.3f}")
            display(pd.DataFrame(
                confusion_matrix(y_test, predictions, labels=labels),
                index=pd.Index(labels, name="actual"),
                columns=pd.Index(labels, name="predicted"),
            ))
            display(pd.DataFrame(classification_report(
                y_test, predictions, labels=labels, output_dict=True, zero_division=0
            )).T)
            names = model.named_steps["preprocessing"].get_feature_names_out()
            importances = model.named_steps["classifier"].feature_importances_
            display(pd.Series(importances, index=names).sort_values(ascending=False)
                    .head(10).rename("importance").to_frame())

forecast_results = pd.DataFrame(rows).set_index(["horizon", "target"])
display(forecast_results.round(3))

## How to read the results

- **Compare with persistence, not with 100%.** `accuracy_gain_vs_persistence` above zero means the Random Forest adds something beyond "tomorrow looks like today". Near zero or negative means it does not.
- **Expect skill to fall from next-day to next-week.** Air quality is only weakly predictable a week out without weather forecasts, so next-week scores close to persistence or climatology are normal, not a bug.
- **No weather of the target day is used.** Real operational forecasts would feed in a weather forecast for day *t+h*. Adding that is a natural next step, and it must be the *forecast* available at time *t*, not the observed weather.
- **Test period is one contiguous block.** It covers only part of a seasonal cycle, so treat the numbers as indicative. A rolling-origin check (`TimeSeriesSplit`) over several cutoffs would be a more robust estimate.
- **Label provenance caveat from the earlier notebook still applies:** some hourly values behind daily labels and features were filled by short linear interpolation (see `sets.ipynb`).
- **"Next week" here means the category 7 days ahead.** If you instead want the category of the *mean AQI over the next 7 days*, only the target construction in section 5 changes.